# Phase 5, Step 1: CodeT5-small on the fixed v2 data (train, evaluate, compare)

One notebook, same structure as Phase 4. It trains `Salesforce/codet5-small` with the Phase 3 and Phase 4 recipe, generates and scores the v2 test set with the exact Phase 4 code, rescores the saved Phase 3 and Phase 4 predictions with the same function, and builds the three-model tables.

Matched to Phase 3 and Phase 4: data (`splits_v2`), seed 42, 3 epochs, effective batch 32 (16 x 2), bf16, 5% warmup (computed as `warmup_steps`), weight decay 0.01, linear schedule, learning rate 5e-5 (same as the CodeT5+ full fine-tune), 512 input and 128 output tokens, `pad_to_multiple_of=8`, 2 dataloader workers, beam search with 4 beams, 128 max new tokens, `early_stopping=True`, A100 GPU.

Run order: every cell, top to bottom, in one session. After a runtime restart, run Setup, Config, and everything below.

**Sections**
1. Setup
2. Config
3. Data loading and checks
4. Tokenizer and model
5. Tokenization
6. Training
7. Save model and training summary
8. Generation (Phase 4 code)
9. Scoring (Phase 4 code)
10. Rescore Phase 3 and Phase 4 predictions, sanity check
11. Three-model tables, length check, efficiency
12. Save summary

## 1. Setup (Drive mount first)

In [ ]:
!pip install -q transformers datasets accelerate evaluate sacrebleu rouge_score nltk sentencepiece

from google.colab import drive
drive.mount('/content/drive')

import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

import torch
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print('GPU:', GPU_NAME)
assert 'A100' in GPU_NAME, 'Efficiency numbers only compare fairly on an A100. Switch the runtime.'

## 2. Config

In [ ]:
import os, json, random, time, math, glob, re
import numpy as np
import pandas as pd
import torch
import transformers

# ---------- Paths ----------
BASE_DIR    = '/content/drive/MyDrive/capstone'
SPLIT_DIR   = f'{BASE_DIR}/data/splits_v2'
RUN_NAME    = 'codet5_small_full_ft_v2_final'
OUTPUT_DIR  = f'{BASE_DIR}/runs/{RUN_NAME}'            # new directory for every new run
RESULTS_DIR = f'{BASE_DIR}/results/phase5_small_final'
PHASE3_DIR  = f'{BASE_DIR}/results/phase3_v2'
PHASE4_DIR  = f'{BASE_DIR}/results/phase4'
LOCAL_MODEL_DIR = '/content/codet5_small_local'

# ---------- Dataset schema ----------
SOURCE_COL, TARGET_COL = 'input_text', 'target_text'

# ---------- Model and training (matched to Phase 3 and Phase 4) ----------
MODEL_NAME = 'Salesforce/codet5-small'
MAX_SOURCE_LEN, MAX_TARGET_LEN = 512, 128
SEED = 42
LEARNING_RATE = 5e-5
NUM_EPOCHS = 3
PER_DEVICE_TRAIN_BS, PER_DEVICE_EVAL_BS = 16, 32
GRAD_ACCUM_STEPS = 2            # effective batch = 32
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LR_SCHEDULER = 'linear'
USE_BF16 = True

# ---------- Generation (identical to Phase 3 and Phase 4) ----------
NUM_BEAMS, MAX_NEW_TOKENS, GEN_BATCH_SIZE = 4, 128, 32

# Guard: a non-empty OUTPUT_DIR makes the Trainer resume instead of training fresh
if os.path.exists(OUTPUT_DIR) and len(os.listdir(OUTPUT_DIR)) > 0:
    raise RuntimeError(f'{OUTPUT_DIR} is not empty. Change RUN_NAME to start a fresh run.')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

## 3. Data loading and checks

In [ ]:
train_df = pd.read_parquet(f'{SPLIT_DIR}/train.parquet')
val_df   = pd.read_parquet(f'{SPLIT_DIR}/val.parquet')
test_df  = pd.read_parquet(f'{SPLIT_DIR}/test.parquet')

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
assert (len(train_df), len(val_df), len(test_df)) == (159902, 19988, 19988), 'Not the v2 splits. Check SPLIT_DIR.'

for d in (train_df, val_df, test_df):
    d['group'] = ['gitbugs' if s == 'gitbugs' else l for s, l in zip(d['source'], d['language'])]
print(test_df['group'].value_counts())

# Leak check on Python inputs (should be well under 1% after the fix)
py = test_df[test_df['group'] == 'python']
leak = np.mean([t.strip() in x for t, x in zip(py[TARGET_COL], py[SOURCE_COL])])
print(f'Python target-in-input rate on test: {leak:.4%}')
assert leak < 0.01, 'Leak detected. This is not the fixed v2 data.'

## 4. Tokenizer and model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainerCallback
from huggingface_hub import snapshot_download

def load_tokenizer_safe(model_name, cache_root='/content/model_cache'):
    # Same helper as the Phase 4 notebook: drop added_tokens.json and normalize special_tokens_map.json
    local_dir = os.path.join(cache_root, model_name.replace('/', '__'))
    snapshot_download(repo_id=model_name, local_dir=local_dir)

    added_tokens_path = os.path.join(local_dir, 'added_tokens.json')
    if os.path.exists(added_tokens_path):
        os.remove(added_tokens_path)

    special_tokens_path = os.path.join(local_dir, 'special_tokens_map.json')
    if os.path.exists(special_tokens_path):
        with open(special_tokens_path, 'r', encoding='utf-8') as f:
            special_tokens = json.load(f)
        additional = special_tokens.get('additional_special_tokens')
        if additional and isinstance(additional[0], dict):
            special_tokens['additional_special_tokens'] = [
                t['content'] if isinstance(t, dict) else t for t in additional
            ]
            with open(special_tokens_path, 'w', encoding='utf-8') as f:
                json.dump(special_tokens, f)

    return AutoTokenizer.from_pretrained(local_dir), local_dir

tokenizer, local_path = load_tokenizer_safe(MODEL_NAME)
print('Tokenizer:', type(tokenizer).__name__, '| vocab size:', len(tokenizer))

model = AutoModelForSeq2SeqLM.from_pretrained(local_path, dtype=torch.float32)   # fp32 weights avoid the sdpa overflow error
print('Weights dtype:', next(model.parameters()).dtype)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total: {total_params:,} | Trainable: {trainable_params:,} ({trainable_params/total_params:.3%})')

## 5. Tokenization

In [ ]:
from datasets import Dataset

def preprocess(batch):
    model_inputs = tokenizer(batch[SOURCE_COL], max_length=MAX_SOURCE_LEN, truncation=True)
    labels = tokenizer(text_target=batch[TARGET_COL], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

def to_ds(df):
    ds = Dataset.from_pandas(df[[SOURCE_COL, TARGET_COL]].reset_index(drop=True), preserve_index=False)
    return ds.map(preprocess, batched=True, remove_columns=[SOURCE_COL, TARGET_COL])

tok_train, tok_val = to_ds(train_df), to_ds(val_df)
print(tok_train, tok_val)

## 6. Training

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

class EfficiencyCallback(TrainerCallback):
    def __init__(self):
        self.epoch_times, self.epoch_peak_mem_gb = [], []
        self._t0 = None
        self.train_start = None
    def on_train_begin(self, args, state, control, **kw):
        self.train_start = time.time()
        torch.cuda.reset_peak_memory_stats()
    def on_epoch_begin(self, args, state, control, **kw):
        self._t0 = time.time()
    def on_epoch_end(self, args, state, control, **kw):
        self.epoch_times.append(time.time() - self._t0)
        self.epoch_peak_mem_gb.append(torch.cuda.max_memory_allocated() / 1e9)

eff_cb = EfficiencyCallback()

collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)

steps_per_epoch = math.ceil(len(tok_train) / (PER_DEVICE_TRAIN_BS * GRAD_ACCUM_STEPS))
WARMUP_STEPS = int(WARMUP_RATIO * steps_per_epoch * NUM_EPOCHS)
print(f'Steps/epoch: {steps_per_epoch:,} | total: {steps_per_epoch * NUM_EPOCHS:,} | warmup steps: {WARMUP_STEPS:,}')

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BS,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    bf16=USE_BF16 and torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_strategy='steps',
    logging_steps=100,
    predict_with_generate=False,
    dataloader_num_workers=2,
    report_to='none',
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=collator,
    processing_class=tokenizer,
    callbacks=[eff_cb],
)

train_result = trainer.train()
print(train_result.metrics)

train_min_sum = sum(eff_cb.epoch_times) / 60
peak_gb = max(eff_cb.epoch_peak_mem_gb)
print(f'Epoch times (s): {[round(t, 1) for t in eff_cb.epoch_times]}')
print(f'Training time (sum of epochs): {train_min_sum:.1f} min | peak GPU memory: {peak_gb:.2f} GB')
print('Best checkpoint:', trainer.state.best_model_checkpoint, '| best eval_loss:', trainer.state.best_metric)

## 7. Save model and loss curves

In [ ]:
BEST_DIR = f'{OUTPUT_DIR}/best'
trainer.save_model(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)

weights = [f for f in os.listdir(BEST_DIR) if f.startswith(('model.safetensors', 'pytorch_model'))]
model_size_mb = round(sum(os.path.getsize(os.path.join(BEST_DIR, f)) for f in weights) / 1e6, 1)
print('Saved to', BEST_DIR, f'| weights size: {model_size_mb} MB')

import matplotlib.pyplot as plt
log = trainer.state.log_history
train_steps  = [l['step'] for l in log if 'loss' in l and 'eval_loss' not in l]
train_losses = [l['loss'] for l in log if 'loss' in l and 'eval_loss' not in l]
eval_steps   = [l['step'] for l in log if 'eval_loss' in l]
eval_losses  = [l['eval_loss'] for l in log if 'eval_loss' in l]

plt.figure(figsize=(8, 4.5))
plt.plot(train_steps, train_losses, label='Train loss', alpha=0.8)
plt.plot(eval_steps, eval_losses, marker='o', label='Validation loss')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('CodeT5-small full fine-tune: loss curves')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_curves_small.png', dpi=200)
plt.show()

pd.DataFrame(log).to_csv(f'{RESULTS_DIR}/log_history_small.csv', index=False)

## 8. Generation (same code as Phase 4)

In [ ]:
device = 'cuda'
model = trainer.model.to(device).eval()

@torch.no_grad()
def generate_all(texts):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    outputs = [None] * len(texts)
    for start in range(0, len(order), GEN_BATCH_SIZE):
        idx = order[start:start + GEN_BATCH_SIZE]
        batch = tokenizer([texts[i] for i in idx], max_length=MAX_SOURCE_LEN, truncation=True,
                          padding=True, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=(device == 'cuda' and USE_BF16)):
            gen = model.generate(**batch, num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS, early_stopping=True)
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        for i, d in zip(idx, decoded):
            outputs[i] = d.strip()
    return outputs

test_sources = test_df[SOURCE_COL].tolist()
test_refs    = test_df[TARGET_COL].tolist()

t0 = time.time()
test_preds = generate_all(test_sources)
gen_seconds = time.time() - t0
lat_ms = 1000 * gen_seconds / len(test_preds)
print(f'Generated {len(test_preds):,} predictions in {gen_seconds/60:.1f} min ({lat_ms:.1f} ms/example)')

## 9. Scoring (same code as Phase 4)

In [ ]:
import evaluate

bleu_m   = evaluate.load('bleu')
rouge_m  = evaluate.load('rouge')
meteor_m = evaluate.load('meteor')

def first_sentence(text):
    t = text.strip().split('\n\n')[0]
    t = ' '.join(t.split())
    m = re.search(r'(?<=[.!?])\s', t)
    return (t[:m.start()] if m else t).strip()

def score(preds, refs):
    preds = [p if p.strip() else ' ' for p in preds]
    refs  = [r if r.strip() else ' ' for r in refs]
    b = bleu_m.compute(predictions=preds, references=[[r] for r in refs], max_order=4)['bleu']
    r = rouge_m.compute(predictions=preds, references=refs)['rougeL']
    m = meteor_m.compute(predictions=preds, references=refs)['meteor']
    return {'BLEU-4': 100 * b, 'ROUGE-L': 100 * r, 'METEOR': 100 * m}

def score_table(df):
    # df needs columns: reference, prediction, group
    d = df.copy()
    d['reference_fs']  = d['reference'].map(first_sentence)
    d['prediction_fs'] = d['prediction'].map(first_sentence)
    variants = [('full_docstring', 'prediction', 'reference'),
                ('first_sentence', 'prediction_fs', 'reference_fs')]
    rows = []
    def add(label, sub):
        for variant, p, r in variants:
            rows.append({'group': label, 'variant': variant, 'n': len(sub),
                         **score(sub[p].tolist(), sub[r].tolist())})
    add('ALL', d)
    for g, sub in d.groupby('group'):
        add(g, sub)
    return pd.DataFrame(rows).round(2)

df = pd.DataFrame({'source': test_df['source'].tolist(), 'reference': test_refs, 'prediction': test_preds})
df['group'] = test_df['group'].tolist()

results = score_table(df)
results.to_csv(f'{RESULTS_DIR}/metrics_small.csv', index=False)
df.to_csv(f'{RESULTS_DIR}/predictions_small.csv', index=False)
results

## 10. Rescore Phase 3 and Phase 4 predictions with the same function

Sanity check first: the rescored full fine-tune and LoRA numbers must reproduce the recorded ALL, full docstring scores (16.40 / 39.89 / 37.54 and 13.32 / 37.79 / 35.29). If either line says MISMATCH, do not trust the tables below.

In [ ]:
def find_one(directory, pattern):
    hits = sorted(glob.glob(os.path.join(directory, pattern)))
    assert hits, f'No file matching {pattern} in {directory}'
    return hits[0]

p3_pred_path = find_one(PHASE3_DIR, 'predictions_full_ft*.csv')
p4_pred_path = find_one(PHASE4_DIR, 'predictions_lora*.csv')
print('Using:', p3_pred_path, '|', p4_pred_path)

p3_df = pd.read_csv(p3_pred_path).fillna('')
p4_df = pd.read_csv(p4_pred_path).fillna('')
p3_results = score_table(p3_df)
p4_results = score_table(p4_df)

recorded = {'full_ft': (16.40, 39.89, 37.54), 'lora': (13.32, 37.79, 35.29)}
for name, res in [('full_ft', p3_results), ('lora', p4_results)]:
    row = res[(res.group == 'ALL') & (res.variant == 'full_docstring')].iloc[0]
    got = (float(row['BLEU-4']), float(row['ROUGE-L']), float(row['METEOR']))
    ok = all(abs(g - r) < 0.05 for g, r in zip(got, recorded[name]))
    print(f'{name}: rescored {got} vs recorded {recorded[name]} -> {"OK" if ok else "MISMATCH"}')

## 11. Three-model tables, length check, efficiency

In [ ]:
all_res = pd.concat([
    results.assign(model='codet5_small'),
    p3_results.assign(model='codet5p_full_ft'),
    p4_results.assign(model='codet5p_lora'),
])
metric_cols = ['BLEU-4', 'ROUGE-L', 'METEOR']

overall = (all_res[all_res.group == 'ALL']
           .pivot_table(index='variant', columns='model', values=metric_cols)
           .round(2))
overall.to_csv(f'{RESULTS_DIR}/three_model_overall.csv')
display(overall)

by_group = (all_res[all_res.variant == 'full_docstring']
            .pivot_table(index='group', columns='model', values='ROUGE-L').round(2))
by_group.to_csv(f'{RESULTS_DIR}/three_model_rougeL_by_group.csv')
display(by_group)

In [ ]:
# Prediction length check: mean words per prediction and per reference, by group
def mean_words(frame, col):
    return frame[col].map(lambda t: len(str(t).split())).groupby(frame['group']).mean()

len_tbl = pd.DataFrame({
    'reference': mean_words(df, 'reference'),
    'small': mean_words(df, 'prediction'),
    'full_ft': mean_words(p3_df, 'prediction'),
    'lora': mean_words(p4_df, 'prediction'),
})
len_tbl.loc['ALL'] = [df['reference'].map(lambda t: len(t.split())).mean(),
                      df['prediction'].map(lambda t: len(t.split())).mean(),
                      p3_df['prediction'].map(lambda t: len(str(t).split())).mean(),
                      p4_df['prediction'].map(lambda t: len(str(t).split())).mean()]
len_tbl = len_tbl.round(1)
len_tbl.to_csv(f'{RESULTS_DIR}/prediction_length_check.csv')
display(len_tbl)

In [ ]:
p3_sum = json.load(open(find_one(PHASE3_DIR, 'summary_full_ft*.json')))
p4_sum = json.load(open(find_one(PHASE4_DIR, 'summary_lora*.json')))
e3, e4 = p3_sum['efficiency'], p4_sum['efficiency']

eff = pd.DataFrame({
    'codet5_small': {
        'trainable params': trainable_params,
        'train time, sum of epochs (min)': round(train_min_sum, 1),
        'peak GPU memory (GB)': round(peak_gb, 2),
        'latency ms/example (batched, full test set)': round(lat_ms, 1),
        'saved size (MB)': model_size_mb,
        'best val loss': round(trainer.state.best_metric, 4),
    },
    'codet5p_full_ft': {
        'trainable params': e3['trainable_params'],
        'train time, sum of epochs (min)': round(sum(e3['epoch_times_sec']) / 60, 1),
        'peak GPU memory (GB)': e3['peak_gpu_mem_gb'],
        'latency ms/example (batched, full test set)': e3['generation_ms_per_example'],
        'saved size (MB)': 890,   # approximate, from the handoff file
        'best val loss': round(p3_sum['best_eval_loss'], 4),
    },
    'codet5p_lora': {
        'trainable params': e4['trainable_params'],
        'train time, sum of epochs (min)': round(sum(e4['epoch_times_sec']) / 60, 1),
        'peak GPU memory (GB)': e4['peak_gpu_mem_gb'],
        'latency ms/example (batched, full test set)': e4['generation_ms_per_example_merged'],
        'saved size (MB)': e4['adapter_size_mb'],
        'best val loss': round(p4_sum['best_eval_loss'], 4),
    },
})
eff.to_csv(f'{RESULTS_DIR}/three_model_efficiency.csv')
print('Efficiency rows are only comparable if all three runs used the same GPU type (A100).')
eff

## 12. Save summary

In [ ]:
row = lambda variant: results[(results.group == 'ALL') & (results.variant == variant)].iloc[0]
summary_small = {
    'model': MODEL_NAME,
    'method': 'full_ft',
    'data': 'splits_v2',
    'hyperparameters': {
        'learning_rate': LEARNING_RATE, 'epochs': NUM_EPOCHS,
        'effective_batch_size': PER_DEVICE_TRAIN_BS * GRAD_ACCUM_STEPS,
        'warmup_ratio': WARMUP_RATIO, 'weight_decay': WEIGHT_DECAY,
        'max_source_len': MAX_SOURCE_LEN, 'max_target_len': MAX_TARGET_LEN,
        'bf16': USE_BF16, 'seed': SEED,
    },
    'generation': {'num_beams': NUM_BEAMS, 'max_new_tokens': MAX_NEW_TOKENS,
                   'early_stopping': True, 'batch_size': GEN_BATCH_SIZE},
    'scoring': 'Phase 4 score() via Hugging Face evaluate (bleu max_order=4, rougeL, meteor)',
    'test_all_full_docstring': {k: float(row('full_docstring')[k]) for k in metric_cols},
    'test_all_first_sentence': {k: float(row('first_sentence')[k]) for k in metric_cols},
    'efficiency': {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'epoch_times_sec': [round(t, 1) for t in eff_cb.epoch_times],
        'train_time_min_sum_of_epochs': round(train_min_sum, 1),
        'peak_gpu_mem_gb': round(peak_gb, 2),
        'generation_ms_per_example': round(lat_ms, 1),
        'saved_size_mb': model_size_mb,
    },
    'best_eval_loss': trainer.state.best_metric,
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'gpu': GPU_NAME,
    'package_versions': {'torch': torch.__version__, 'transformers': transformers.__version__},
}
with open(f'{RESULTS_DIR}/summary_small.json', 'w') as f:
    json.dump(summary_small, f, indent=2)
print(json.dumps(summary_small, indent=2))
print('\nSaved to', RESULTS_DIR)